# Persist Read Server Examples

Examples for reading locally mirrored `uniform_orders` from the persist read server on this machine.

Sources configured here:

- `okex-intra-arb01`
- `binance-intra-arb01`
- `bybit-intra-arb01`


In [1]:
import time
import urllib.parse
import urllib.request
from datetime import datetime, timezone

import pandas as pd
import pyarrow as pa
import pyarrow.ipc as ipc

BASE_URL = 'http://127.0.0.1:8822'
SOURCES = ['okex-intra-arb01', 'binance-intra-arb01', 'bybit-intra-arb01']
TABLE = 'uniform_orders'


## Helpers


In [ ]:
def get_json(path, params=None, timeout=30):
    url = BASE_URL + path
    if params:
        url += '?' + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        return __import__('json').loads(resp.read().decode('utf-8'))


def read_arrow(table, source_id, start_us, end_us, columns=None, timeout=120):
    params = {
        'table': table,
        'source_id': source_id,
        'start_us': int(start_us),
        'end_us': int(end_us),
        'format': 'arrow_ipc',
    }
    if columns:
        params['columns'] = ','.join(columns) if isinstance(columns, (list, tuple)) else columns
    url = BASE_URL + '/v1/read?' + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        data = resp.read()
    return ipc.open_stream(pa.BufferReader(data)).read_all()


def read_uniform_orders(source_id, start_us, end_us, columns=None):
    # columns=None -> 不带 columns 参数请求，服务端返回全部 22 列
    table = read_arrow(TABLE, source_id, start_us, end_us, columns)
    return table.to_pandas()


def utc_window(hours=1):
    end_us = int(time.time() * 1_000_000)
    start_us = end_us - int(hours * 3600 * 1_000_000)
    return start_us, end_us


def fmt_us(ts_us):
    return datetime.fromtimestamp(ts_us / 1_000_000, timezone.utc).isoformat()


## Health And Schema


In [4]:
print(get_json('/healthz'))
for source in SOURCES:
    schema = get_json('/v1/schema', {'table': TABLE, 'source_id': source})
    print(source, schema['table'], len(schema['columns']), schema['formats'])


{'ok': True}
okex-intra-arb01 uniform_orders 22 ['arrow_ipc', 'parquet']
binance-intra-arb01 uniform_orders 22 ['arrow_ipc', 'parquet']
bybit-intra-arb01 uniform_orders 22 ['arrow_ipc', 'parquet']


## Latest 1 Hour Row Counts


In [5]:
start_us, end_us = utc_window(hours=1)
print('UTC window:', fmt_us(start_us), '->', fmt_us(end_us))

summary = []
for source in SOURCES:
    df = read_uniform_orders(source, start_us, end_us, columns=['ts_us', 'symbol', 'client_order_id', 'trading_venue', 'status'])
    summary.append({'source_id': source, 'rows': len(df)})
pd.DataFrame(summary)


UTC window: 2026-06-11T04:10:15.780904+00:00 -> 2026-06-11T05:10:15.780904+00:00


,source_id,rows
0,okex-intra-arb01,4116
1,binance-intra-arb01,10
2,bybit-intra-arb01,2157


## Pull Each Source


In [5]:
okex_orders = read_uniform_orders('okex-intra-arb01', start_us, end_us)
okex_orders.head()


,ts_us,symbol,client_order_id,trading_venue,side,price,amount_init,amount_update,status
0,1781150848572405,ETHUSDT,2841939551495127041,OkexMargin,BUY,1652.60,0.03,0.030,FILLED
1,1781150848574224,ETHUSDT,2841939555790094337,OkexMargin,BUY,1652.48,0.03,0.000,CANCELED
2,1781150848576398,ETHUSDT,2840696729693586429,OkexFutures,SELL,0.00,0.03,0.000,NEW
3,1781150848576412,ETHUSDT,2841939560085061633,OkexMargin,BUY,1652.37,0.03,0.000,CANCELED
4,1781150848576693,ETHUSDT,2840696729693586429,OkexFutures,SELL,1651.63,0.03,0.004,PARTIALLY_FILLED


In [6]:
binance_orders = read_uniform_orders('binance-intra-arb01', start_us, end_us)
binance_orders.head()


,ts_us,symbol,client_order_id,trading_venue,side,price,amount_init,amount_update,status
0,1781152946046462,KNCUSDT,2629588620259885057,BinanceMargin,SELL,0.1223,408.0,84.1,PARTIALLY_FILLED
1,1781153097979094,KNCUSDT,2629588620259885057,BinanceMargin,SELL,0.1223,408.0,323.9,FILLED
2,1781153097982477,KNCUSDT,2628931292695101516,BinanceFutures,BUY,0.0000,408.0,0.0,NEW
3,1781153097984229,KNCUSDT,2628931292695101516,BinanceFutures,BUY,0.1222,408.0,408.0,FILLED
4,1781153445615986,KNCUSDT,2629362984152989697,BinanceMargin,SELL,0.1227,407.0,407.0,FILLED


In [7]:
bybit_orders = read_uniform_orders('bybit-intra-arb01', start_us, end_us)
bybit_orders.head()


,ts_us,symbol,client_order_id,trading_venue,side,price,amount_init,amount_update,status
0,1781150853186110,RENDERUSDT,3061383964817096705,BybitMargin,BUY,1.5550,32.1,0.0,NEW
1,1781150854732868,SPXUSDT,3061383758658666497,BybitMargin,BUY,0.3207,155.0,0.0,CANCELED
2,1781150854865493,ICPUSDT,3061384012061736961,BybitMargin,SELL,2.2870,21.8,0.0,NEW
3,1781150857129438,AVAXUSDT,3061384063601344513,BybitMargin,BUY,6.5760,7.6,0.0,NEW
4,1781150857436761,TONUSDT,3061383724298928129,BybitMargin,BUY,1.6510,30.2,0.0,CANCELED


## Combined Analysis Example


In [ ]:
# 逐盘拉默认列，前面插入 source_id 区分来源，最后 concat 成一张总表
frames = []
for source in SOURCES:
    df = read_uniform_orders(source, start_us, end_us)
    df.insert(0, 'source_id', source)
    frames.append(df)
orders = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
# ts_us 是微秒整数；加一列 UTC tz-aware 的时间戳，方便 resample / groupby('1min') 之类
orders['ts'] = pd.to_datetime(orders['ts_us'], unit='us', utc=True)
orders.head()


In [ ]:
# 各盘按订单状态计数：NEW / PARTIALLY_FILLED / FILLED / CANCELED / EXPIRED
# 用来看成交率、撤单率，对比不同盘子的执行特征
orders.groupby(['source_id', 'status']).size().reset_index(name='orders').sort_values(['source_id', 'orders'], ascending=[True, False])


In [ ]:
# 每个盘子下单量最大的品种 Top 20，用来看活跃 symbol 分布
orders.groupby(['source_id', 'symbol']).size().reset_index(name='orders').sort_values('orders', ascending=False).head(20)


## Bybit 期现套利订单：从 2026-06-05 08:00:00 (北京) 起完整抓取

- **数据源**：`bybit-intra-arb01` —— bybit 期现套利策略产出的订单流。订单中 `trading_venue` 字段会出现 `BybitMargin`/`BybitSpot`（现货腿）和 `BybitFutures`（期货腿），同一 `client_order_id` 的两条腿即一组期现配对。
- **表**：`uniform_orders`
- **时间区间**：北京时间 `2026-06-05 08:00:00` → 当前时刻（= UTC `2026-06-05 00:00:00` → now）
- **完整性保证**：服务端单次窗口对返回大小有限制（直接一把拉会 HTTP 400），所以下面按 `chunk_minutes` 滚动分块拉，相邻窗口左闭右开 `[cur, nxt)` 无重叠无遗漏，最后按 `ts_us` 排序去重。

In [ ]:
from datetime import datetime, timezone, timedelta

BEIJING = timezone(timedelta(hours=8))

def fetch_orders_range(source_id, start_beijing, end_beijing=None, chunk_minutes=30):
    """
    从 persist_read 服务按时间区间完整拉取某个 source 的 uniform_orders。
    start_beijing/end_beijing: 'YYYY-MM-DD HH:MM:SS'（北京时间），end 不传则取当前 UTC。
    分块滚动拉取以绕过单窗口返回上限；区间 [cur, nxt) 左闭右开，无重叠。
    """
    start_dt = datetime.strptime(start_beijing, '%Y-%m-%d %H:%M:%S').replace(tzinfo=BEIJING)
    end_dt = (datetime.strptime(end_beijing, '%Y-%m-%d %H:%M:%S').replace(tzinfo=BEIJING)
              if end_beijing else datetime.now(timezone.utc))
    assert end_dt > start_dt, 'end must be after start'

    step = timedelta(minutes=chunk_minutes)
    cur, frames, failed = start_dt, [], []
    while cur < end_dt:
        nxt = min(cur + step, end_dt)
        s_us, e_us = int(cur.timestamp() * 1_000_000), int(nxt.timestamp() * 1_000_000)
        try:
            df = read_uniform_orders(source_id, s_us, e_us)
            if len(df):
                frames.append(df)
        except Exception as ex:
            failed.append((cur, nxt, str(ex)))
        cur = nxt

    if failed:
        print(f'[WARN] {len(failed)} chunk(s) failed, first: {failed[0]}')
    if not frames:
        return pd.DataFrame()

    out = (pd.concat(frames, ignore_index=True)
             .drop_duplicates(subset=['ts_us', 'client_order_id', 'status'])
             .sort_values('ts_us')
             .reset_index(drop=True))
    out['ts_bj'] = pd.to_datetime(out['ts_us'], unit='us', utc=True).dt.tz_convert('Asia/Shanghai')
    return out


# === 北京时间 2026-06-05 08:00:00 起，bybit 期现套利全部订单 ===
START_BJ = '2026-06-05 08:00:00'
bybit_arb_orders = fetch_orders_range('bybit-intra-arb01', START_BJ, chunk_minutes=30)

print('source     :', 'bybit-intra-arb01  (table: uniform_orders)')
print('range (BJ) :', START_BJ, '->', datetime.now(BEIJING).strftime('%Y-%m-%d %H:%M:%S'))
print('rows       :', len(bybit_arb_orders))
if len(bybit_arb_orders):
    print('first ts   :', bybit_arb_orders['ts_bj'].iloc[0])
    print('last  ts   :', bybit_arb_orders['ts_bj'].iloc[-1])
    print('venues     :', bybit_arb_orders['trading_venue'].value_counts().to_dict())
bybit_arb_orders.head()
